In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.optim import Adam
from torchvision.transforms import v2
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from PIL import Image

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

In [2]:
LR = 1e-4
BATCH_SIZE = 32
EPOCHS = 100

In [3]:
rand_seed = 42
DIR = "dataset"
image_path = []
label = []

for image in os.listdir(DIR):
    image_path.append(f"{DIR}/{image}")
    if "dog" in image:
        label.append(0)  # 0 for dogs
    else:
        label.append(1)  # 1 for cats

data_df = pd.DataFrame(zip(image_path, label), columns = ["image_path", "label"])
data_df = data_df.sample(frac=1, random_state=rand_seed)
train_df = data_df.sample(frac=0.7)
test_df = data_df.drop(train_df.index)
val_df = test_df.sample(frac=0.5)
test_df = test_df.drop(val_df.index)

In [4]:
# transform convert image to the same size, file type and property 
transform = v2.Compose([
    v2.Resize((128, 128)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True)
]) 

class Dataset(Dataset): 
    '''
    Custom image dataset must inherit from Dataset so that DataLoader can cooperate with it
    '''
    def __init__(self, dataframe, transform = None): 
        self.dataframe = dataframe
        self.transform = transform # data preprocessing function from torch vision
        self.labels = torch.tensor(dataframe["label"].values)

    def __len__(self):
        return self.dataframe.shape[0]

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx, 0]
        label = self.labels[idx]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

train_dataset = Dataset(dataframe=train_df, transform=transform)
val_dataset = Dataset(dataframe=val_df, transform=transform)
test_dataset = Dataset(dataframe=test_df, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [5]:
def get_one_batch(train_df, transform, batch_size):
    ds = Dataset(train_df, transform=transform)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)
    for batch in dl:
        return batch

m, n = get_one_batch(train_df, transform, BATCH_SIZE)
print(m.shape)

torch.Size([32, 3, 128, 128])


In [13]:
class CNN(nn.Module):
    def __init__(self, pool_size=(2, 2), num_classes=1, dropout_p=0.5):
        super().__init__()

        # declaring the layers (3 channels (rgb), output 32 channels)
        self.conv1 = nn.Conv2d(3, 32, kernel_size = 3, padding = 1)
        self.bn1 = nn.BatchNorm2d(num_features=32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size = 3, padding = 1)
        self.bn2 = nn.BatchNorm2d(num_features=64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size = 3, padding = 1) 
        self.bn3 = nn.BatchNorm2d(num_features=128)

        self.pooling = nn.MaxPool2d(kernel_size=pool_size)
        self.dropout = nn.Dropout(p=dropout_p)
        self.flatten = nn.Flatten()
        # self.linear = nn.Linear((128 * 14 * 14), 128) # without padding
        self.linear = nn.Linear((128 * 16 * 16), 128) # with padding
        self.output = nn.Linear(128, num_classes)

    def forward(self, x):
        '''
        Foward (conv -> bn -> relu -> pool -> dropout)
        '''
        x = self.conv1(x) # (3, 128, 128) -> (32, 126, 126) (without padding
        # pooling reduce the size, but keeps the features
        x = self.bn1(x)
        x = F.relu(x) # use nn.functional is better ... for some reason
        # conv increase # of features, but does not change the size
        x = self.pooling(x) # -> (32, 63, 63) 
        x = self.dropout(x)

        x = self.conv2(x) # -> (64, 61, 61) (without padding
        x = self.bn2(x)
        x = F.relu(x) # activation function
        x = self.pooling(x) # -> (64, 30, 30)
        x = self.dropout(x)

        x = self.conv3(x) # -> (128, 28, 28) (without padding
        x = self.bn3(x)
        x = F.relu(x) # activation function
        x = self.pooling(x) # -> (128, 14, 14) -> final conv result 
        x = self.dropout(x)

        x = self.flatten(x)
        x = self.linear(x)
        x = F.relu(x)
        x = self.output(x)

        # use BCEWithLogitsLoss instead of sigmoid function
        # x = self.sigmoid(x)

        return x

model = CNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = Adam(model.parameters(), lr=LR)

In [7]:
from torchsummary import summary
summary(model, input_size = (3, 128, 128))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 128, 128]             896
       BatchNorm2d-2         [-1, 32, 128, 128]              64
         MaxPool2d-3           [-1, 32, 64, 64]               0
           Dropout-4           [-1, 32, 64, 64]               0
            Conv2d-5           [-1, 64, 64, 64]          18,496
       BatchNorm2d-6           [-1, 64, 64, 64]             128
         MaxPool2d-7           [-1, 64, 32, 32]               0
           Dropout-8           [-1, 64, 32, 32]               0
            Conv2d-9          [-1, 128, 32, 32]          73,856
      BatchNorm2d-10          [-1, 128, 32, 32]             256
        MaxPool2d-11          [-1, 128, 16, 16]               0
          Dropout-12          [-1, 128, 16, 16]               0
          Flatten-13                [-1, 32768]               0
           Linear-14                  [

In [14]:
min_val_loss = float("inf")
patience = 7

for epoch in range(EPOCHS):
    if patience == 0:
        print("Training stopped due to loss of patience.")
        break

    total_acc_train = 0
    total_loss_train = 0
    total_loss_val = 0
    total_acc_val = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(torch.float32).to(device)
        optimizer.zero_grad()
        outputs = model(inputs).squeeze()
        train_loss = criterion(outputs, labels)
        total_loss_train += train_loss.item() 
        train_loss.backward()
        train_acc = ((outputs > 0.0).float() == labels).sum().item()
        total_acc_train += train_acc
        optimizer.step()

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(torch.float32).to(device)
            outputs = model(inputs).squeeze()
            val_loss = criterion(outputs, labels)
            total_loss_val += val_loss.item()
            val_acc = ((outputs > 0.0).float() == labels).sum().item()
            total_acc_val += val_acc

    total_loss_train = total_loss_train/train_dataset.__len__()
    total_loss_val = total_loss_val/val_dataset.__len__()
    total_acc_train = total_acc_train/train_dataset.__len__() * 100
    total_acc_val = total_acc_val/val_dataset.__len__() * 100

    print(f'''
Epoch: {epoch+1},
Train Loss: {total_loss_train},
Validation Loss: {total_loss_val},
Train Acc: {total_acc_train},
Validation Acc:{total_acc_val}
    ''')

    if total_loss_val < min_val_loss:
        torch.save(model.state_dict(), "model.pth")
        min_val_loss = total_loss_val
        print("Model saved!")
        patience = 7  # restore patience
    else:
        patience -= 1


Epoch: 1,
Train Loss: 0.021689604350498746,
Validation Loss: 0.021118431985378265,
Train Acc: 57.17857142857142,
Validation Acc:59.166666666666664
    
Model saved!

Epoch: 2,
Train Loss: 0.019818027168512345,
Validation Loss: 0.019980058073997498,
Train Acc: 64.35714285714286,
Validation Acc:62.83333333333333
    
Model saved!

Epoch: 3,
Train Loss: 0.01971334580864225,
Validation Loss: 0.020117171009381613,
Train Acc: 65.85714285714286,
Validation Acc:63.33333333333333
    

Epoch: 4,
Train Loss: 0.019352221893412725,
Validation Loss: 0.019658085306485495,
Train Acc: 66.89285714285714,
Validation Acc:65.33333333333333
    
Model saved!

Epoch: 5,
Train Loss: 0.018821400052734784,
Validation Loss: 0.018739445606867473,
Train Acc: 67.17857142857143,
Validation Acc:68.16666666666666
    
Model saved!

Epoch: 6,
Train Loss: 0.018311898495469773,
Validation Loss: 0.01948636531829834,
Train Acc: 69.57142857142857,
Validation Acc:64.5
    

Epoch: 7,
Train Loss: 0.017595794062529292,
Valid